# C3: Keys & Strings Business Assistant

EECE 503P/798S - Introduction to Agents.

This notebook explains and runs the chatbot using the shared implementation in
`business_agent.py` and `app.py`. It covers the business profile, persona,
instructions, router, tools, conversation history and Gradio interface.

Install the dependencies in `requirements.txt`, select the repository's `.venv`
Python kernel and run the cells in order from the repository root. Configure your
API key privately in `.env` for chat. The final cell launches the app.
See `README.md` for setup and
[the demo video](demo/MokhtarNasser_keys%26strings_demo.mp4).


## 1. Setup and business evidence


In [ ]:
import inspect
import json
import logging
import os
import sys
from pathlib import Path
from IPython.display import Code, display
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "business_agent.py").is_file():
    raise RuntimeError("Start this notebook from the repository root.")
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from business_agent import (
    BusinessAgent, KnowledgeBase, PERSONA, INSTRUCTIONS, ROUTER_PROMPT,
    ROUTE_RESPONSE_FORMAT, TOOL_SCHEMAS,
    record_customer_interest, record_feedback,
)
from app import create_demo

knowledge = KnowledgeBase()
print(f"Loaded {len(knowledge.products)} core products and {len(knowledge.accessories['items'])} accessories.")
print(knowledge.summary)
print("PDF identity excerpt:", knowledge.profile[:550])
print("Catalogue date:", knowledge.catalogue["as_of"])


The PDF and summary describe the business. Structured JSON supplies product
facts and store policies. Answers always receive a short business overview,
product categories and opening hours. Local search adds up to six matching
product records, accessories and relevant PDF or policy excerpts. Inventory
lists can also use compact price and stock rows.


In [ ]:
print(json.dumps(knowledge.retrieve(["KS-001", "KS-020"]), indent=2, ensure_ascii=False))
print("Delivery rules:", json.dumps(knowledge.policies["delivery"], indent=2))


## 2. Persona — identity and voice


In [ ]:
# Identity and tone only. The same constant is used by the running app.
print(PERSONA)


## 3. Operational instructions — factual and behavioral rules


In [ ]:
# Separate from the persona: grounding, uncertainty, lead collection, and tools.
print(INSTRUCTIONS)


## 4. Compact LLM router and local retrieval

The router classifies a message as `general_question`, `lead_collection` or
`feedback`. It requests strict JSON output with a 64-token cap and uses up to
four recent messages of 400 characters each. Invalid output stops the turn.
Asking to leave feedback first requires collecting its contents.

Retrieval runs locally, matching model names, model-code aliases, brands,
budgets and recent references. Accessory matching handles singular/plural names.


In [ ]:
print("Classifier instructions:", ROUTER_PROMPT)
print("Output schema:", json.dumps(ROUTE_RESPONSE_FORMAT, indent=2))
display(Code(inspect.getsource(BusinessAgent.route), language="python"))


## 5. Required tools and schemas

Exact signatures: `record_customer_interest(email, name, message)` and
`record_feedback(question)`. These append UTF-8 JSONL with a record UUID and UTC
timestamp. Results contain `success`, `record_id`, and `file`, or `success: false`
and `error`. Input validation rejects blank fields and invalid email syntax.
The dispatcher uses an allowlist and exact field checks. Logs default to the
gitignored `logs/` directory. Neither tool sends email or creates a booking.
See `docs/TOOLS.md` for the full contract and error behavior.


In [ ]:
display(Code(inspect.getsource(record_customer_interest), language="python"))
display(Code(inspect.getsource(record_feedback), language="python"))
print(json.dumps(TOOL_SCHEMAS, indent=2))


## 6. OpenAI chat and short-term history

Routing and answering use `client.chat.completions.create(...)`. The default
model is `gpt-6-luna` with `reasoning_effort="none"` and low verbosity. Routing is
capped at 64 output tokens; each answer is capped at 1,024. A normal turn makes
two requests, with additional answering requests after tool results.

The answer prompt combines the persona, operating rules and selected business
evidence. History retains up to 16 recent messages, capped at 12,000 characters
total and 1,000 per historical message. New messages and tool notes are limited
to 2,000 characters.

Each tool result is returned with its matching `tool_call_id`. Identical calls
execute once per turn. The loop allows two tool rounds, followed by a final
request with tools disabled. A successful write is required before confirming
that a record was saved.


In [ ]:
display(Code(inspect.getsource(BusinessAgent.respond), language="python"))
assistant = BusinessAgent(knowledge=knowledge)  # lazy API client; no request yet


## 7. Gradio ChatInterface

Run the next cell to launch the app. Chat uses OpenAI with your configured key.
Set `C3_LAUNCH_UI=0` for a headless notebook run. Stop with `demo.close()` before
relaunching. Use `GRADIO_SERVER_PORT` if the default port is occupied.

[Demo video](demo/MokhtarNasser_keys%26strings_demo.mp4)


In [ ]:
demo = create_demo(assistant)
if os.getenv("C3_LAUNCH_UI", "1") == "1":
    demo.launch(server_name="127.0.0.1", server_port=int(os.getenv("GRADIO_SERVER_PORT", "7860")),
                share=False, inline=True, prevent_thread_lock=True)
else:
    print("Gradio ChatInterface constructed; server launch disabled for headless checking.")
